In [1]:
import sys,os
sys.path.append(r'C:/Users/andrej/Projects/EnergyTrading/Python/Strategies/LeadLagXGB/')
sys.path.append(r'Z:/EnergyTrading/Python/')
sys.path.append(r'Z:/EnergyTrading/Python/Strategies/LeadLagXGB/')
from support_functions import TR_class, calculate_MACD, calculate_lead_lag_triggers, calculate_regression_model_price, calc_vol_intensity_index

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

In [3]:
def get_trades_for_contract(contract, start_date, end_date):
    params_dict={}
    params_dict['tenor_list'] = ['dec'] if contract=='euadec1' else [contract[-2]]
    params_dict['tn1_list'] = [int(contract[-1])]
    params_dict['mkt_list'] = ['eua'] * len(params_dict['tenor_list']) if contract=='euadec1' else [contract[0:-2]] * len(params_dict['tenor_list'])
    params_dict['tn2_list'] = []
    params_dict['prod'] = 'base'
    params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
    params_dict['start_date'] = start_date
    params_dict['end_date'] = end_date
    params_dict['ns'] = 2

    # Fetch trades and best orders for the curve
    assembler = TPDataAssembly(source='database', user='matej')
    # assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
    trades_dict = assembler.get_data(params_dict, target_data='trades')
    #assembler.set_data_source('database')
    #ba_dict = assembler.get_data(params_dict, target_data='best_orders')


    trades = pd.DataFrame()
    products = []
    for key in trades_dict.keys():
        trade_aux = trades_dict[key].copy()
        trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
        if trades.empty:
            trades = trade_aux.copy()
        else:
            trades = pd.concat([trades, trade_aux])
        products.append(key)
    trades.sort_index(inplace=True)




    data_raw = trades
    print(data_raw.columns)
    data_raw['tradeid_'+contract]=data_raw['tradeid_'+contract].apply(lambda x: str(x)[:-7] if str(x)[-7:]==' Public' else str(x))
    df_lead = data_raw[data_raw['broker_id_'+contract]==1441][['tradeid_'+contract,'price_'+contract, 'volume_'+contract]].copy()
    
    df_lead['contract']=contract

    # data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
    #                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

    df_lead.columns = [a.split('_')[0] for a in df_lead.columns]
    print(df_lead.columns)
    df_lead.columns = ['tradeid', 'trd_price', 'volume', 'contract']
    
    return df_lead





In [4]:
start_date='2025-01-01'
end_date='2025-04-28'

# Selecting all lead and lag EEX trades since 2025

In [5]:
contracts=['de_frm1', 'esm1', 'frm1', 'frq1', 'fry1', 'itm1', 'itq1', 'dem1', 'dey1', 'dey2',  'deq1', 'dem2', 'ttfm1']
contracts

['de_frm1',
 'esm1',
 'frm1',
 'frq1',
 'fry1',
 'itm1',
 'itq1',
 'dem1',
 'dey1',
 'dey2',
 'deq1',
 'dem2',
 'ttfm1']

In [6]:
df_all=pd.concat([get_trades_for_contract(contract, start_date, end_date) for contract in contracts])

Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Index(['tradeid_de_frm1', 'price_de_frm1', 'volume_de_frm1', 'action_de_frm1',
       'broker_id_de_frm1', 'own_trades_de_frm1'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Index(['tradeid_esm1', 'price_esm1', 'volume_esm1', 'action_esm1',
       'broker_id_esm1', 'own_trades_esm1'],
      dtype='object')
Index(['tradeid', 'price', 'volume', 'contract'], dtype='object')
Connected to t

In [7]:
df_all['contract'].value_counts()

contract
dem1       261564
ttfm1      107849
dem2        95298
deq1        73135
dey1        44368
frm1        36329
itm1        12172
de_frm1     11569
frq1         9990
dey2         7273
itq1         6291
fry1         4124
esm1         1893
Name: count, dtype: int64

In [8]:
df_all=df_all.reset_index()

In [9]:
df_all['datetime'] = pd.to_datetime(df_all['index'], errors='coerce')
df_all['timestamp'] = df_all['datetime']

In [10]:
df_all=df_all[df_all['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]

In [11]:
df_all.head()

,index,tradeid,trd_price,volume,contract,datetime,timestamp
0,2025-01-02 09:42:52.209270467,Eurex T7/DEBMF7BM022025-20250102/1483/1,10.99,1,de_frm1,2025-01-02 09:42:52.209270467,2025-01-02 09:42:52.209270467
1,2025-01-02 09:47:04.669153595,Eurex T7/DEBMF7BM022025-20250102/1588/1,10.51,1,de_frm1,2025-01-02 09:47:04.669153595,2025-01-02 09:47:04.669153595
2,2025-01-02 10:59:01.932985651,Eurex T7/DEBMF7BM022025-20250102/3097/1,10.37,1,de_frm1,2025-01-02 10:59:01.932985651,2025-01-02 10:59:01.932985651
3,2025-01-02 11:14:16.470471811,Eurex T7/DEBMF7BM022025-20250102/3348/1,10.01,1,de_frm1,2025-01-02 11:14:16.470471811,2025-01-02 11:14:16.470471811
4,2025-01-02 11:14:16.470471811,Eurex T7/DEBMF7BM022025-20250102/3349/1,10.00,1,de_frm1,2025-01-02 11:14:16.470471811,2025-01-02 11:14:16.470471811


In [12]:
df_all.set_index('datetime', inplace=True)

## Ticking

In [13]:
def tick_contract(contract):
    sort_order=[True, False, True]
    print(contract)
    df_trds=df_all[df_all['contract']==contract]

    agg_dict = {'index': 'first', 'datetime': 'first', 'trd_price': 'mean'}

    df_trds['date'] = df_trds.index.date
    dates_list = sorted(list(set(df_trds['date'])))

    df_list = []
    df_list2 = []

    for current_day in dates_list:
        df_trds_1day = df_trds[df_trds['date'] == current_day].reset_index()
        df_trds_1day['execution_time'] = df_trds_1day['datetime'].astype('int64')  # Already in nanoseconds

        # Preparing tick data
        ti_cls = TR_class(10, 10)

        df_trds_1day = df_trds_1day.sort_values(by=['datetime', 'volume', 'trd_price'], ascending=sort_order).reset_index()


        # Create the list of tuples
        lag_trades = [(row['trd_price'], row['volume'], row['execution_time']) for index, row in
                      df_trds_1day.iterrows()]


        idx_series = ti_cls.tick_imbalance_single(lag_trades)
        idx_series = pd.DataFrame(idx_series, columns=['index', 0])


        if 'level_0' in df_trds_1day.columns:
            del df_trds_1day['level_0']

        # Create returns
        df_trds_indexed_1day = pd.concat([df_trds_1day, idx_series[0]], axis=1).reset_index()

        df_trds_indexed_1day['tick_id'] = str(current_day) + '_' + df_trds_indexed_1day[0].fillna(0).apply(str)
        #df_trds_indexed_1day['datetime'] = df_trds_indexed_1day['timestamp']
        #df_trds_indexed_1day = df_trds_indexed_1day.set_index('datetime')
        df_trds_indexed_1day_grouped = df_trds_indexed_1day.groupby('tick_id').agg(
            agg_dict).reset_index().set_index('datetime')

        df_trds_indexed_1day_grouped['log_ret'] = np.log(df_trds_indexed_1day_grouped['trd_price'].ffill() ).diff()

        df_list.append(df_trds_indexed_1day_grouped)
        df_list2.append(df_trds_indexed_1day)

    df_trds_indexed = pd.concat(df_list).sort_index()
    
    df_trds_indexed['contract']=contract
    
    return df_trds_indexed

In [14]:
#df_all=pd.concat([tick_contract(contract) for contract in contracts])

In [15]:
df_filtered = df_all[df_all['contract'].apply(lambda x: x in ['dem1', 'dey1', 'dey2',  'deq1', 'dem2', 'ttfm1'])].dropna(subset=['log_return']).copy()
df_filtered['date'] = df_filtered.index.normalize()

KeyError: ['log_return']

# Shannon entropies

## Daily

In [ ]:
import numpy as np
from scipy.stats import entropy

bins=10

def shannon_entropy(data, bins=bins):
    data = data.dropna()
    if len(data) < 2:
        return np.nan
    hist, _ = np.histogram(data, bins=bins, density=True)
    hist = hist[hist > 0]
    return entropy(hist, base=2)

# Compute log returns
df_all['log_return'] = np.log(df_all['trd_price']).diff()

In [ ]:
daily_entropy = (
    df_filtered.groupby(['contract', 'date'])['log_return']
    .apply(shannon_entropy)
    .reset_index()
    .rename(columns={'log_return': 'daily_entropy'})
)

In [ ]:
pivot_df = daily_entropy.pivot(index='date', columns='contract', values='daily_entropy')

In [ ]:
import matplotlib.pyplot as plt

max_entropy = np.log2(bins)

plt.figure(figsize=(14, 6))
pivot_df.plot(ax=plt.gca(), linewidth=1.2)
plt.axhline(max_entropy, color='black', linestyle='--', label=f"Max Entropy ≈ {max_entropy:.2f} bits")

plt.title("Daily Shannon Entropy by Contract")
plt.xlabel("Date")
plt.ylabel("Entropy (bits)")
plt.legend(title="Contract", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


## Weekly

In [ ]:
weekly_avg_entropy = pivot_df.resample('W').mean()

In [ ]:
import matplotlib.pyplot as plt

max_entropy = np.log2(bins)

plt.figure(figsize=(14, 6))
weekly_avg_entropy.plot(ax=plt.gca(), linewidth=1.5)
plt.axhline(max_entropy, color='black', linestyle='--', label=f"Max Entropy ≈ {max_entropy:.2f} bits")

plt.title("Weekly Shannon Entropy by Contract")
plt.xlabel("Week")
plt.ylabel("Entropy (bits)")
plt.legend(title="Contract", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Monthly

In [ ]:
monthly_avg_entropy = pivot_df.resample('M').mean()

In [ ]:
import matplotlib.pyplot as plt

bins = 10
max_entropy = np.log2(bins)

plt.figure(figsize=(14, 6))
monthly_avg_entropy.plot(ax=plt.gca(), linewidth=1.5)
plt.axhline(max_entropy, color='black', linestyle='--', label=f"Max Entropy ≈ {max_entropy:.2f} bits")

plt.title("Monthly Shannon Entropy by Contract")
plt.xlabel("Month")
plt.ylabel("Entropy (bits)")
plt.legend(title="Contract", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

# Permutation entropies

## Daily

In [ ]:
from antropy import perm_entropy

def safe_perm_entropy(x, order=4, delay=1):
    x = x.dropna().values
    if len(x) < order + 1:
        return np.nan
    return perm_entropy(x, order=order, delay=delay, normalize=True)

# Group by contract and date
pe_df = (
    df_filtered.groupby(['contract', 'date'])['log_return']
    .apply(safe_perm_entropy)
    .reset_index(name='perm_entropy')
)

In [ ]:
pivot_pe = pe_df.pivot(index='date', columns='contract', values='perm_entropy')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
pivot_pe.plot(ax=plt.gca())
plt.title("Daily Permutation Entropy by Contract")
plt.ylabel("Normalized Permutation Entropy")
plt.xlabel("Date")
plt.legend(title="Contract", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Weekly

In [ ]:
weekly_avg_entropy = pivot_pe.resample('W').mean()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
weekly_avg_entropy.plot(ax=plt.gca(), linewidth=1.5)

plt.title("Weekly Shannon Entropy by Contract")
plt.xlabel("Week")
plt.ylabel("Entropy (bits)")
plt.legend(title="Contract", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Monthly

In [ ]:
monthly_avg_entropy = pivot_pe.resample('M').mean()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
monthly_avg_entropy.plot(ax=plt.gca(), linewidth=1.5)

plt.title("Monthly Shannon Entropy by Contract")
plt.xlabel("Month")
plt.ylabel("Entropy (bits)")
plt.legend(title="Contract", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()